In this estimatedsalry will be output feature


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
df=pd.read_csv('Churn_Modelling.csv')

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## preprocess the data
## drop irrelvent features
df=df.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [5]:
## encoding categorical variable
label_encoder_gender=LabelEncoder()
df['Gender']=label_encoder_gender.fit_transform(df['Gender'])

In [6]:
## Onehot encoding for geography column
from sklearn.preprocessing import OneHotEncoder
onehot = OneHotEncoder(sparse_output=False)  # Ensure it returns a dense array

# Fit-transform the data
encoded_values = onehot.fit_transform(df[['Geography']])

# Retrieve correct column names from OneHotEncoder
encoded_columns = onehot.get_feature_names_out(['Geography'])

# Convert the NumPy array to a DataFrame with the correct column names
encoded_df = pd.DataFrame(encoded_values, columns=encoded_columns)

# Concatenate with original DataFrame (optional)
df = pd.concat([df, encoded_df], axis=1)

In [7]:
df=df.drop(columns=['Geography'])

In [8]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
## divide the data set into independent an d dependent
x=df.drop('EstimatedSalary', axis=1)
y=df['EstimatedSalary']

x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=.2, random_state=42)



In [10]:
x_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
9254,686,1,32,6,0.00,2,1,1,0,1.0,0.0,0.0
1561,632,1,42,4,119624.60,2,1,1,0,0.0,1.0,0.0
1670,559,1,24,3,114739.92,1,1,0,1,0.0,0.0,1.0
6087,561,0,27,9,135637.00,1,1,0,1,1.0,0.0,0.0
6669,517,1,56,9,142147.32,1,0,0,1,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5734,768,1,54,8,69712.74,1,1,1,0,1.0,0.0,0.0
5191,682,0,58,1,0.00,1,1,1,0,1.0,0.0,0.0
5390,735,0,38,1,0.00,3,0,0,1,1.0,0.0,0.0
860,667,1,43,8,190227.46,1,1,0,1,1.0,0.0,0.0


In [11]:
## scle these feature
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [12]:
x_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [13]:
## save the encoder ans scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot, file)  

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)

In [14]:
## train aNN  regression problem statement
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(x_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

# compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

c:\Users\ashis\Desktop\python\deep learning\venv\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,033 (47.00 KB)

 Trainable params: 12,033 (47.00 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
## set up the tensorboard
# from dat
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [27]:
## setup early stopping- it is used to moderate the loss value, if it not decreasing , then early stopping stop the procedure
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
 

In [28]:
## training the model
history=model.fit(
    x_train,y_train, validation_data=(x_test,y_test), epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100241.2891 - mae: 100241.2891 - val_loss: 87756.8203 - val_mae: 87756.8203
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 75843.7656 - mae: 75843.7656 - val_loss: 50547.7344 - val_mae: 50547.7344
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49755.6055 - mae: 49755.6055 - val_loss: 50259.5586 - val_mae: 50259.5586
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50275.1719 - mae: 50275.1719 - val_loss: 50204.7344 - val_mae: 50204.7344
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49720.5352 - mae: 49720.5352 - val_loss: 50195.7656 - val_mae: 50195.7656
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50649.2773 - mae: 50649.2773 - val_loss: 50289.2617 - val_mae: 50289.2617
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49618.2500 - mae: 49618.2500 - val_loss: 50195.3203 - val_mae: 50195.3203
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

In [29]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [30]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 24904), started 0:24:15 ago. (Use '!kill 24904' to kill it.)

In [31]:
## Evfaluate model on test data
test_loss, test_mae=model.evaluate(x_test,y_test)
print(f'test mae : {test_mae}')
print(f'test loss : {test_loss}')

52/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1000us/step - loss: 51271.9062 - mae: 51271.9062

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 51079.7656 - mae: 51079.7656   
test mae : 50195.3203125
test loss : 50195.3203125


In [32]:
model.save('regression_model.h5')